In [0]:
import glob, os, re
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

CATALOGO, SCHEMA = "workspace", "default"
VOL = f"/Volumes/{CATALOGO}/{SCHEMA}/kelmarsh_raw"
ANO_DIR = f"{VOL}/scada_2018"

print("arquivos enviados: ")
for f in sorted(os.listdir(ANO_DIR)):
  print(f)

arquivos enviados: 
Status_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
Status_Kelmarsh_2_2018-01-01_-_2019-01-01_229.csv
Status_Kelmarsh_3_2018-01-01_-_2019-01-01_230.csv
Status_Kelmarsh_4_2018-01-01_-_2019-01-01_231.csv
Status_Kelmarsh_5_2018-01-01_-_2019-01-01_232.csv
Status_Kelmarsh_6_2018-01-01_-_2019-01-01_233.csv
Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
Turbine_Data_Kelmarsh_2_2018-01-01_-_2019-01-01_229.csv
Turbine_Data_Kelmarsh_3_2018-01-01_-_2019-01-01_230.csv
Turbine_Data_Kelmarsh_4_2018-01-01_-_2019-01-01_231.csv
Turbine_Data_Kelmarsh_5_2018-01-01_-_2019-01-01_232.csv
Turbine_Data_Kelmarsh_6_2018-01-01_-_2019-01-01_233.csv


todos os arquivos esperados subiram, documentado neste notebook

In [0]:
# higienizar nome das colunas, já observado que possui caracteres especiais
# vou padronizar com uma funcao

def limpar_nome(nome):
    nome = nome.strip().lower() #sempre minuscula e sem espaços sobrando
    caracteres_especiais = " !@#$%¨&*()_+{}[]|\/?><.,;:'"
    for caracter in caracteres_especiais:
        nome = nome.replace(caracter, "_") #substituo pelo underscore
    while "__" in nome:
        nome = nome.replace("__", "_") #manter um underscore só

    nome = nome.strip("_") #remover _ sobrando antes ou depois

    return nome

print(limpar_nome("Wind speed (m/s)"))

def pegar_turbina(nome_do_arquivo):
    partes = nome_do_arquivo.split("_")     # quebra o nome onde tem "_"
    posicao = partes.index("Kelmarsh")      # acha a palavra "Kelmarsh"
    return partes[posicao + 1]              # o número vem logo depois

print(pegar_turbina("Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv"))  # -> 1

wind_speed_m_s
1


funcao testada e ok

In [0]:
lista = sorted(glob.glob(f"{ANO_DIR}/Turbine_Data_Kelmarsh_*.csv"))
exemplo = lista[0]

N = 9 #skip de linhas de cabecalho do supervisorio ate chegar nos dados

df = pd.read_csv(exemplo, skiprows=N, dtype=str) #dtype=str para evitar conversao automatica, apenas porque eh bronze

nomes_limpos = []
for col in df.columns:
    nomes_limpos.append(limpar_nome(col))
df.columns = nomes_limpos

#conferindo o shape do dataset
print(df.shape) #esperado 52560 linhas por 299 col


(52560, 299)


In [0]:
palavras = ["wind", "power", "rotor", "pitch", "nacelle",
            "temp", "generator", "gearbox", "bearing", "ambient"]

for coluna in df.columns:
    for palavra in palavras:
        if palavra in coluna:
            print(coluna)
            break   # já achou, não precisa testar as outras palavras

wind_speed_m_s
wind_speed_standard_deviation_m_s
wind_speed_minimum_m_s
wind_speed_maximum_m_s
long_term_wind_m_s
wind_speed_sensor_1_m_s
wind_speed_sensor_1_standard_deviation_m_s
wind_speed_sensor_1_minimum_m_s
wind_speed_sensor_1_maximum_m_s
wind_speed_sensor_2_m_s
wind_speed_sensor_2_standard_deviation_m_s
wind_speed_sensor_2_minimum_m_s
wind_speed_sensor_2_maximum_m_s
density_adjusted_wind_speed_m_s
wind_direction_°
nacelle_position_°
wind_direction_standard_deviation_°
wind_direction_minimum_°
wind_direction_maximum_°
nacelle_position_standard_deviation_°
nacelle_position_minimum_°
nacelle_position_maximum_°
power_kw
potential_power_default_pc_kw
power_standard_deviation_kw
power_minimum_kw
power_maximum_kw
potential_power_learned_pc_kw
potential_power_reference_turbines_kw
cascading_potential_power_kw
cascading_potential_power_for_performance_kw
potential_power_met_mast_anemometer_kw
potential_power_primary_reference_turbines_kw
potential_power_secondary_reference_turbines_kw
tu

In [0]:
# ingerir os 6 arquivos de dados para os dados supervisionados das turbinas (SCADA)

arquivos = sorted(glob.glob(f"{ANO_DIR}/Turbine_Data_Kelmarsh_*.csv"))

for i, caminho in enumerate(arquivos):
    nome_do_arquivo = os.path.basename(caminho)
    turbina = pegar_turbina(nome_do_arquivo)

    df = pd.read_csv(caminho, skiprows=N, dtype=str)

    nomes_limpos = []
    for col in df.columns:
        nomes_limpos.append(limpar_nome(col))
    df.columns = nomes_limpos

    df["turbina"] = turbina
    df["arquivo_origem"] = nome_do_arquivo

    # gravar no delta: primeiro recria a tabela os outros acrescentam

    if i==0:
        modo = "overwrite"
    else:
        modo = "append"

    (spark.createDataFrame(df)
     .write.mode(modo)
     .option("mergeSchema", "true")
     .format("delta")
     .saveAsTable(f"{CATALOGO}.{SCHEMA}.bronze_scada"))
    
    print(f"turbina {turbina}: {df.shape[0]} linhas -> {modo}")
    

turbina 1: 52560 linhas -> overwrite
turbina 2: 52560 linhas -> append
turbina 3: 52560 linhas -> append
turbina 4: 52560 linhas -> append
turbina 5: 52560 linhas -> append
turbina 6: 52560 linhas -> append


In [0]:
# ingerindo os dados para os status das turbinas

arquivos = sorted(glob.glob(f"{ANO_DIR}/Status_Kelmarsh_*.csv"))

for i, caminho in enumerate(arquivos):
    nome_do_arquivo = os.path.basename(caminho)
    turbina = pegar_turbina(nome_do_arquivo)

    df = pd.read_csv(caminho, skiprows=N, dtype=str)

    nomes_limpos = []
    for col in df.columns:
        nomes_limpos.append(limpar_nome(col))
    df.columns = nomes_limpos

    df["turbina"] = turbina
    df["arquivo_origem"] = nome_do_arquivo

    # gravar no delta: primeiro recria a tabela os outros acrescentam

    if i==0:
        modo = "overwrite"
    else:
        modo = "append"

    (spark.createDataFrame(df)
     .write.mode(modo)
     .option("mergeSchema", "true")
     .saveAsTable(f"{CATALOGO}.{SCHEMA}.bronze_status"))
    
    print(f"turbina {turbina}: {df.shape[0]} linhas -> {modo}")


turbina 1: 10674 linhas -> overwrite
turbina 2: 9443 linhas -> append
turbina 3: 12515 linhas -> append
turbina 4: 9933 linhas -> append
turbina 5: 11444 linhas -> append
turbina 6: 15954 linhas -> append


dados raw ingeridos e pre-processados

In [0]:
print("SCADA (esperado 52560 linhas por turbina):")
display(spark.sql(f"""
                  SELECT turbina, COUNT(*) AS linhas
                  FROM {CATALOGO}.{SCHEMA}.bronze_scada
                  GROUP BY turbina
                  ORDER BY turbina
                  """))
print("STATUS (esperado alguns milhares de linhas por turbina):")
display(spark.sql(f"""
                  SELECT turbina, COUNT (*) AS eventos
                  FROM {CATALOGO}.{SCHEMA}.bronze_status
                  GROUP BY turbina
                  ORDER BY turbina
                  """
                  ))

SCADA (esperado 52560 linhas por turbina):


turbina,linhas
1,52560
2,52560
3,52560
4,52560
5,52560
6,52560


STATUS (esperado alguns milhares de linhas por turbina):


turbina,eventos
1,10674
2,9443
3,12515
4,9933
5,11444
6,15954


In [0]:
# selecionando as features adequadas para o projeto de acordo com a engenharia

colunas_medidas = [
    "date_and_time",
    "wind_speed_m_s", "wind_speed_standard_deviation_m_s",
    "wind_speed_minimum_m_s", "wind_speed_maximum_m_s",
    "power_kw",
    "rotor_speed_rpm", "generator_rpm_rpm", "blade_angle_pitch_position_a_°",
    "front_bearing_temperature_°c", "rear_bearing_temperature_°c",
    "gear_oil_temperature_°c", "generator_bearing_front_temperature_°c",
    "nacelle_ambient_temperature_°c",
]

display(spark.table(f"{CATALOGO}.{SCHEMA}.bronze_scada").select(colunas_medidas).limit(3))


date_and_time,wind_speed_m_s,wind_speed_standard_deviation_m_s,wind_speed_minimum_m_s,wind_speed_maximum_m_s,power_kw,rotor_speed_rpm,generator_rpm_rpm,blade_angle_pitch_position_a_°,front_bearing_temperature_°c,rear_bearing_temperature_°c,gear_oil_temperature_°c,generator_bearing_front_temperature_°c,nacelle_ambient_temperature_°c
2018-11-16 09:00:00,3.045919418334961,0.31527072191238403,2.49114990234375,3.5501248836517334,1.388478398323059,3.8384783267974854,457.6555480957031,27.31223487854004,41.80277633666992,43.61111068725586,42.71666717529297,41.96666717529297,11.505555152893066
2018-11-16 09:10:00,2.416435956954956,0.26020464301109314,1.9413282871246338,3.0142416954040527,2.7129664421081543,8.996395111083984,1067.5692138671875,1.4900000095367432,44.587501525878906,49.38999938964844,43.64250183105469,41.06999969482422,11.640000343322754
2018-11-16 09:20:00,3.0084125995635986,0.41249045729637146,1.9970163106918335,3.6631858348846436,20.982921600341797,8.995770454406738,1066.435302734375,1.4900000095367432,47.34000015258789,52.477500915527344,44.400001525878906,40.88750076293945,11.694999694824219
